# General Conference Trend Analysis: What Has Changed Over 50 Years?

This notebook systematically identifies:
- **Words** that have increased or decreased the most
- **Phrases** that show the biggest changes
- **Themes/concepts** that have shifted over time
- **Statistical validation** of specific hypotheses

## Hypotheses to Test

1. **More Christ-centered**: Focus on Savior, atonement, charity, love
2. **Less administrative**: Less focus on programs, organization, peculiarity
3. **New vocabulary**: Words like "intentional", "covenant path", "ministering"
4. **Doctrinal shifts**: Changes in emphasis on specific doctrines

In [ ]:
# Setup
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from conference_analysis.scraper import ConferenceScraper
from conference_analysis.trend_analysis import TrendAnalyzer
from conference_analysis.temporal_analysis import TemporalAnalyzer
from conference_analysis.embeddings import EmbeddingsAnalyzer

# Plotting setup
import plotly.io as pio
pio.renderers.default = 'notebook'
sns.set_style('whitegrid')

print("✅ Setup complete!")

In [ ]:
# Load data
scraper = ConferenceScraper(cache_file='../data/raw/talks.csv')
talks = scraper.talks_df

print(f"Loaded {len(talks)} talks")
print(f"Date range: {talks['date'].min()} to {talks['date'].max()}")

In [ ]:
# Initialize analyzers
trend_analyzer = TrendAnalyzer(talks)
temporal_analyzer = TemporalAnalyzer(talks)

print("✅ Analyzers ready!")

## Part 1: Systematic Discovery - What Has Changed?

Let's systematically find the words and phrases with the biggest changes.

In [ ]:
# Compare 1970s (1971-1979) vs 2020s (2020-2024)
early_period = (1971, 1979)
late_period = (2020, 2024)

print(f"Comparing {early_period[0]}-{early_period[1]} vs {late_period[0]}-{late_period[1]}")

### Top 30 Words That Have INCREASED

In [ ]:
increasing = trend_analyzer.find_increasing_words(early_period, late_period, top_n=30)

print("\n" + "="*80)
print("TOP 30 WORDS THAT HAVE INCREASED THE MOST")
print("="*80)
print(f"{'Word':<20} {'1970s Freq':<15} {'2020s Freq':<15} {'Change':<12} {'% Change':<12}")
print("-"*80)

for idx, row in increasing.iterrows():
    print(f"{row['word']:<20} {row['early_freq']:<15.2f} {row['late_freq']:<15.2f} {row['change']:<12.2f} {row['pct_change']:<12.1f}%")

# Display as dataframe for easier viewing
increasing[['word', 'early_freq', 'late_freq', 'change', 'pct_change']].head(30)

### Top 30 Words That Have DECREASED

In [ ]:
decreasing = trend_analyzer.find_decreasing_words(early_period, late_period, top_n=30)

print("\n" + "="*80)
print("TOP 30 WORDS THAT HAVE DECREASED THE MOST")
print("="*80)
print(f"{'Word':<20} {'1970s Freq':<15} {'2020s Freq':<15} {'Change':<12} {'% Change':<12}")
print("-"*80)

for idx, row in decreasing.iterrows():
    print(f"{row['word']:<20} {row['early_freq']:<15.2f} {row['late_freq']:<15.2f} {row['change']:<12.2f} {row['pct_change']:<12.1f}%")

decreasing[['word', 'early_freq', 'late_freq', 'change', 'pct_change']].head(30)

### Visualize Top Changes

In [ ]:
# Create visualization
fig = trend_analyzer.visualize_word_changes(early_period, late_period, top_n=20)
fig.show()

### Top Phrases That Have Changed

Now let's look at 2-3 word phrases (bigrams and trigrams).

**Smart stopword filtering:**
- **Bigrams (2 words)**: Rejects if EITHER word is a stopword
  - ✅ "brothers sisters" (both meaningful)
  - ❌ "brothers and" (contains stopword)
- **Trigrams (3+ words)**: Rejects if FIRST or LAST word is a stopword
  - ✅ "brothers and sisters" (meaningful words at edges)
  - ❌ "and sisters in" (stopword at edge)

This ensures you see meaningful phrases like "covenant path" and "brothers and sisters", not fragments like "brothers and" or "of the".

In [ ]:
# Find phrases with biggest changes
# Note: filter_stopwords=True by default (removes unhelpful phrases)
# Set filter_stopwords=False to see all phrases including stopword fragments

phrase_changes = trend_analyzer.compare_phrases(
    early_period, 
    late_period, 
    ngram_range=(2, 3),  # 2-3 word phrases
    top_n=40,
    min_occurrences=5,
    filter_stopwords=True  # Smart filtering (default)
)

print("\n" + "="*100)
print("TOP PHRASES WITH BIGGEST CHANGES (Stopwords Filtered)")
print("="*100)

# Increasing phrases
print("\nINCREASING PHRASES:")
print("-"*100)
increasing_phrases = phrase_changes[phrase_changes['change'] > 0].head(20)
for idx, row in increasing_phrases.iterrows():
    ngram_label = f"({row['ngram_size']}-gram)" if 'ngram_size' in row else ""
    print(f"{row['phrase']:<40} {ngram_label:<10} {row['early_freq']:>8.2f} → {row['late_freq']:>8.2f} ({row['change']:>+8.2f})")

# Decreasing phrases
print("\nDECREASING PHRASES:")
print("-"*100)
decreasing_phrases = phrase_changes[phrase_changes['change'] < 0].head(20)
for idx, row in decreasing_phrases.iterrows():
    ngram_label = f"({row['ngram_size']}-gram)" if 'ngram_size' in row else ""
    print(f"{row['phrase']:<40} {ngram_label:<10} {row['early_freq']:>8.2f} → {row['late_freq']:>8.2f} ({row['change']:>+8.2f})")

## Part 2: Testing Specific Hypotheses

Let's test your specific hypotheses about how conference talks have changed.

### Hypothesis 1: More Christ-Centered Language

In [ ]:
christ_centered_words = [
    'christ', 'savior', 'atonement', 'redeemer', 
    'mediator', 'grace', 'mercy', 'charity', 'love'
]

results = trend_analyzer.test_hypothesis(
    christ_centered_words, 
    early_period, 
    late_period
)

print("\n" + "="*80)
print("HYPOTHESIS: More Christ-Centered Language")
print("="*80)
print(f"{'Word':<15} {'1970s Freq':<15} {'2020s Freq':<15} {'Change':<12} {'% Change'}")
print("-"*80)

for word, data in results.items():
    print(f"{word:<15} {data['early_freq']:<15.2f} {data['late_freq']:<15.2f} "
          f"{data['change']:<12.2f} {data['pct_change']:>+10.1f}%")

# Overall trend
total_early = sum(d['early_freq'] for d in results.values())
total_late = sum(d['late_freq'] for d in results.values())
total_change = total_late - total_early
pct_change = (total_change / total_early * 100) if total_early > 0 else 0

print("="*80)
print(f"{'TOTAL':<15} {total_early:<15.2f} {total_late:<15.2f} "
      f"{total_change:<12.2f} {pct_change:>+10.1f}%")
print("\n✅ CONFIRMED" if total_change > 0 else "❌ NOT CONFIRMED")

### Hypothesis 2: Less Administrative/Programmatic Language

In [ ]:
administrative_words = [
    'program', 'programs', 'organization', 'auxiliary', 
    'committee', 'procedure', 'administration', 'correlation'
]

results = trend_analyzer.test_hypothesis(
    administrative_words, 
    early_period, 
    late_period
)

print("\n" + "="*80)
print("HYPOTHESIS: Less Administrative/Programmatic Language")
print("="*80)
print(f"{'Word':<15} {'1970s Freq':<15} {'2020s Freq':<15} {'Change':<12} {'% Change'}")
print("-"*80)

for word, data in results.items():
    print(f"{word:<15} {data['early_freq']:<15.2f} {data['late_freq']:<15.2f} "
          f"{data['change']:<12.2f} {data['pct_change']:>+10.1f}%")

total_early = sum(d['early_freq'] for d in results.values())
total_late = sum(d['late_freq'] for d in results.values())
total_change = total_late - total_early
pct_change = (total_change / total_early * 100) if total_early > 0 else 0

print("="*80)
print(f"{'TOTAL':<15} {total_early:<15.2f} {total_late:<15.2f} "
      f"{total_change:<12.2f} {pct_change:>+10.1f}%")
print("\n✅ CONFIRMED" if total_change < 0 else "❌ NOT CONFIRMED")

### Hypothesis 3: New Vocabulary ("intentional", "covenant path", etc.)

In [ ]:
new_vocabulary = [
    'intentional', 'covenant path', 'ministering', 
    'authentic', 'journey', 'minister'
]

# Note: For phrases like "covenant path", we'll search for the phrase directly
print("\n" + "="*80)
print("HYPOTHESIS: New Vocabulary Emergence")
print("="*80)
print(f"{'Word/Phrase':<20} {'1970s Freq':<15} {'2020s Freq':<15} {'Change'}")
print("-"*80)

for term in new_vocabulary:
    results = trend_analyzer.test_hypothesis([term], early_period, late_period)
    data = results[term]
    print(f"{term:<20} {data['early_freq']:<15.2f} {data['late_freq']:<15.2f} "
          f"{data['change']:>+12.2f}")

## Part 3: Comparative Concept Analysis

Compare multiple concepts over ALL decades to see the full trend.

In [ ]:
# Define concept groups
concepts = {
    'Christ-centered': ['christ', 'savior', 'atonement', 'redeemer'],
    'Love & Charity': ['love', 'charity', 'compassion', 'kindness'],
    'Covenant': ['covenant', 'covenants'],
    'Administrative': ['program', 'programs', 'organization', 'auxiliary'],
    'Peculiar People': ['peculiar', 'different', 'distinct', 'unique']
}

fig = trend_analyzer.comparative_concept_analysis(concepts, time_grouping='decade')
fig.show()

### Focus on Specific Words Over Time

In [ ]:
# Track specific words you're interested in
fig = trend_analyzer.visualize_concept_over_time(
    ['intentional'], 
    'Intentional',
    time_grouping='decade'
)
fig.show()

In [ ]:
# Covenant path - look at the phrase
words_to_track = ['covenant path']
fig = temporal_analyzer.plot_phrase_trends(
    words_to_track,
    time_grouping='year',
    normalize=True,
    interactive=True
)
fig.update_layout(title='"Covenant Path" Usage Over Time')
fig.show()

## Part 4: Semantic/Thematic Analysis Using Embeddings

Now let's use embeddings to find CONCEPTUAL changes, not just word changes.

In [ ]:
# Load embeddings
embeddings_analyzer = EmbeddingsAnalyzer(talks, cache_dir='../data/processed')
embeddings_analyzer.generate_embeddings()
print("✅ Embeddings loaded!")

### How "Christ-Centered" Talks Have Evolved

In [ ]:
# Find talks most similar to "Jesus Christ and His atonement" in each decade
time_periods = [
    (1971, 1979),
    (1980, 1989),
    (1990, 1999),
    (2000, 2009),
    (2010, 2019),
    (2020, 2024)
]

christ_evolution = embeddings_analyzer.temporal_semantic_shift(
    concept="Jesus Christ, His atonement, and His role as our Savior",
    time_periods=time_periods,
    top_k=3
)

print("\n" + "="*80)
print("HOW CHRIST-CENTERED TALKS HAVE EVOLVED")
print("="*80)

for period in christ_evolution['period'].unique():
    period_talks = christ_evolution[christ_evolution['period'] == period]
    avg_similarity = period_talks['similarity'].mean()
    
    print(f"\n{period}: Average Similarity = {avg_similarity:.3f}")
    print("-" * 80)
    for idx, row in period_talks.iterrows():
        print(f"  {row['similarity']:.3f} - {row['title']} ({row['speaker']})")

### Find Talks About "Ministry" vs "Administration"

In [ ]:
# Compare ministry/service vs administration across time
ministry_evolution = embeddings_analyzer.temporal_semantic_shift(
    concept="ministering to others, serving, caring for individuals",
    time_periods=time_periods,
    top_k=3
)

admin_evolution = embeddings_analyzer.temporal_semantic_shift(
    concept="church programs, organization, administration",
    time_periods=time_periods,
    top_k=3
)

# Calculate average similarity for each period
ministry_avg = ministry_evolution.groupby('period')['similarity'].mean()
admin_avg = admin_evolution.groupby('period')['similarity'].mean()

print("\n" + "="*80)
print("MINISTRY vs ADMINISTRATION FOCUS")
print("="*80)
print(f"{'Period':<15} {'Ministry Avg':<20} {'Admin Avg':<20} {'Difference'}")
print("-" * 80)

comparison = pd.DataFrame({
    'period': ministry_avg.index,
    'ministry': ministry_avg.values,
    'administration': admin_avg.values
})
comparison['difference'] = comparison['ministry'] - comparison['administration']

for idx, row in comparison.iterrows():
    print(f"{row['period']:<15} {row['ministry']:<20.3f} {row['administration']:<20.3f} "
          f"{row['difference']:>+10.3f}")

print("\nNote: Positive difference = more ministry focus, Negative = more admin focus")

## Part 5: Summary & Insights

Let's create a comprehensive summary visualization.

In [ ]:
# Create comprehensive comparison
all_concepts = {
    'Christ-centered': ['christ', 'savior', 'atonement'],
    'Love & Service': ['love', 'charity', 'ministering', 'minister'],
    'Covenant': ['covenant', 'covenants'],
    'Grace & Mercy': ['grace', 'mercy'],
    'Faith': ['faith', 'belief', 'trust'],
    'Administrative': ['program', 'programs', 'organization'],
}

fig = trend_analyzer.comparative_concept_analysis(all_concepts, time_grouping='decade')
fig.update_layout(
    title='Comprehensive Trend Analysis: Key Concepts Over 50 Years',
    height=700,
    width=1200
)
fig.show()

## Part 6: Export Results

Save your findings for further analysis or presentation.

In [ ]:
# Save top increasing and decreasing words
increasing.to_csv('../data/processed/increasing_words_1970s_vs_2020s.csv', index=False)
decreasing.to_csv('../data/processed/decreasing_words_1970s_vs_2020s.csv', index=False)

# Save phrase changes
phrase_changes.to_csv('../data/processed/phrase_changes_1970s_vs_2020s.csv', index=False)

print("✅ Results saved to data/processed/")

## Your Custom Analysis

Use the cells below to test your own hypotheses!

In [ ]:
# Test your own hypothesis here
# Example:
# my_words = ['example', 'words', 'to', 'test']
# results = trend_analyzer.test_hypothesis(my_words, early_period, late_period)
